In [1]:
# план конспекта: 
# 1) кросс-валидация для задачи регрессии
# 2) кросс-валидация для задачи классификации с дисбалансом классов
# 3) кросс-валидация для задачи классификации с дисбалансом классов с учетом стратфикации
# 4) изучение настраиваемых параметров коробочной полносвязной нейронной сети

In [2]:
# 1) кросс-валидация для задачи регрессии
# рассмотрим старый добрый датасет на регрессию про цены на дома

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as seaborn
from collections import Counter

path = kagglehub.dataset_download("yasserh/housing-prices-dataset")
print("Path to dataset files:", path)
df = pd.read_csv(path+'\\Housing.csv')
df

Path to dataset files: C:\Users\user\.cache\kagglehub\datasets\yasserh\housing-prices-dataset\versions\1


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [3]:
df3 = df

col_list = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for i in col_list:
    df3[i] = df3[i].map({'yes': 1, 'no': 0})

df3['status_furnished'] = df3['furnishingstatus'].map({'furnished': 1}).fillna(0)
df3['status_semi_furnished'] = df3['furnishingstatus'].map({'semi-furnished': 1}).fillna(0)
df3['status_unfurnished'] = df3['furnishingstatus'].map({'unfurnished': 1}).fillna(0)

df3 = df3.drop(['furnishingstatus'], axis = 1)

df3['price'] = df3['price'] / 1e6
df3['area'] = df3['area'] / 1e3

df3

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,status_furnished,status_semi_furnished,status_unfurnished
0,13.30000,7.42,4,2,3,1,0,0,0,1,2,1,1.0,0.0,0.0
1,12.25000,8.96,4,4,4,1,0,0,0,1,3,0,1.0,0.0,0.0
2,12.25000,9.96,3,2,2,1,0,1,0,0,2,1,0.0,1.0,0.0
3,12.21500,7.50,4,2,2,1,0,1,0,1,3,1,1.0,0.0,0.0
4,11.41000,7.42,4,1,2,1,1,1,0,1,2,0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1.82000,3.00,2,1,1,1,0,1,0,0,2,0,0.0,0.0,1.0
541,1.76715,2.40,3,1,1,0,0,0,0,0,0,0,0.0,1.0,0.0
542,1.75000,3.62,2,1,1,1,0,0,0,0,0,0,0.0,0.0,1.0
543,1.75000,2.91,3,1,1,0,0,0,0,0,0,0,1.0,0.0,0.0


In [4]:
from sklearn.model_selection import train_test_split

x = df3.drop(['price'], axis = 1).to_numpy()
y = df3['price'].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

In [12]:
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut, LeavePOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

reg = LinearRegression().fit(x_train, y_train)

reg_score_train = reg.score(x_train, y_train)
reg_score_test = reg.score(x_test, y_test)

#(n_splits - количество групп, на каждой итерации одна из групп будет тестовой (без повторений))
kf = KFold(n_splits = 10, random_state = 42, shuffle = True)

kf_score = cross_val_score(reg, x, y, cv = kf)

kf_score = list(map(lambda x: round(x, 3).item(), kf_score))

print('R^2 для линейной регрессии на выборке для обучения / теста:\t', round(reg_score_train, 3), '/', round(reg_score_test, 3))
print('R^2 на k-fold разбиениях: \t\t\t\t\t', kf_score)
print("Так, на k-fold разбиениях \t\t\t\t\tточность: %0.2f со стандартным отклонением: %0.2f" % (np.array(kf_score).mean(), np.array(kf_score).std()))

R^2 для линейной регрессии на выборке для обучения / теста:	 0.678 / 0.688
R^2 на k-fold разбиениях: 					 [0.688, 0.592, 0.514, 0.721, 0.492, 0.491, 0.758, 0.602, 0.718, 0.507]
Так, на k-fold разбиениях 					точность: 0.61 со стандартным отклонением: 0.10


In [16]:
# LeaveOneOut не работает для R^2 метрики, поэтому используем LeavePOut (n-p строк - на тренеровку, p строк - на тест), n итераций (вычислительно затратно)
#lpo = LeavePOut(10)

#lpo_score = cross_val_score(reg, x, y, cv = lpo)
#lpo_score = list(map(lambda x: round(x, 3).item(), lpo_score))

In [17]:
# таким образом, первоначальная оценка обучения модели (R^2 метрика) удовлетворительная
# однако, кросс-валидация показывает, что модель имеет не лучшие параметры устойчивости
# данное обстоятельство требует (преимущественно) доработки первоначального датасета

In [22]:
# 2) кросс-валидация для задачи классификации с дисбалансом классов
# рассмотрим задачку про одобрение кредитных карт

path = kagglehub.dataset_download("rohitudageri/credit-card-details")
print("Path to dataset files:", path)
df1 = pd.read_csv(path+'\\Credit_card.csv')
df2 = pd.read_csv(path+'\\Credit_card_label.csv')
df4 = df1.merge(df2[['Ind_ID', 'label']], on='Ind_ID', how='left')
df4

Path to dataset files: C:\Users\user\.cache\kagglehub\datasets\rohitudageri\credit-card-details\versions\1


,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2,1
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,5028645,F,N,Y,0,NaN,Commercial associate,Higher education,Married,House / apartment,-11957.0,-2182,1,0,0,0,Managers,2,0
1544,5023655,F,N,N,0,225000.0,Commercial associate,Incomplete higher,Single / not married,House / apartment,-10229.0,-1209,1,0,0,0,Accountants,1,0
1545,5115992,M,Y,Y,2,180000.0,Working,Higher education,Married,House / apartment,-13174.0,-2477,1,0,0,0,Managers,4,0
1546,5118219,M,Y,N,0,270000.0,Working,Secondary / secondary special,Civil marriage,House / apartment,-15292.0,-645,1,1,1,0,Drivers,2,0


In [23]:
df4 = df4.drop(['Ind_ID', 'Birthday_count','Employed_days', 'Mobile_phone', 'Work_Phone', 
                'Phone', 'Type_Occupation', 'EDUCATION', 'Marital_status', 'EMAIL_ID', 'Housing_type'], axis=1)

df4['GENDER'] = df4['GENDER'].map({'M': 1, 'F': 0})
df4 = df4.dropna(subset=['GENDER'])
df4['Car_Owner'] = df4['Car_Owner'].map({'Y': 1, 'N': 0})
df4['Propert_Owner'] = df4['Propert_Owner'].map({'Y': 1, 'N': 0})
df4['Annual_income'] = df4['Annual_income'].fillna(0)
df4['Type_Income_Pensioner'] = df4['Type_Income'].map({'Pensioner': 1}).fillna(0)
df4 = df4.drop('Type_Income', axis=1)

df4['Annual_income'] = df4['Annual_income'] / 1e3

df4

,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Family_Members,label,Type_Income_Pensioner
0,1.0,1,1,0,180.0,2,1,1.0
1,0.0,1,0,0,315.0,2,1,0.0
2,0.0,1,0,0,315.0,2,1,0.0
3,0.0,1,0,0,0.0,2,1,0.0
4,0.0,1,0,0,315.0,2,1,0.0
...,...,...,...,...,...,...,...,...
1543,0.0,0,1,0,0.0,2,0,0.0
1544,0.0,0,0,0,225.0,1,0,0.0
1545,1.0,1,1,2,180.0,4,0,0.0
1546,1.0,1,0,0,270.0,2,0,0.0


In [24]:
Counter(df4['label']) #явный дисбаланс

Counter({0: 1371, 1: 170})

In [26]:
x = df4.drop(['label'], axis = 1).to_numpy()
y = df4['label'].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score

log = LogisticRegression(max_iter = 10000).fit(x_train, y_train)

log_predict_train = log.predict(x_train)
log_predict_test = log.predict(x_test)

log_score_train = recall_score(log_predict_train, y_train, zero_division = 0, average=None)
log_score_test = recall_score(log_predict_test, y_test, zero_division = 0, average=None)

kf = KFold(n_splits = 10, random_state = 42, shuffle = True)

kf_score = cross_val_score(log, x, y, cv = kf)
kf_score = list(map(lambda x: round(x, 3).item(), kf_score))

print('Если не избавляться от дисбаланса классов при обучении, то')
print('recall для логистической регрессии на выборке для обучения / теста:\t', log_score_train, '/', log_score_test)
print('accuracy на k-fold разбиениях: \t\t\t\t\t\t', kf_score)
print("Так, на k-fold разбиениях \t\t\t\t\t\t точность: %0.2f со стандартным отклонением: %0.2f" % (np.array(kf_score).mean(), np.array(kf_score).std()))

Если не избавляться от дисбаланса классов при обучении, то
recall для логистической регрессии на выборке для обучения / теста:	 [0.89033189 0.        ] / [0.88387097 0.        ]
accuracy на k-fold разбиениях: 						 [0.884, 0.929, 0.922, 0.916, 0.825, 0.883, 0.851, 0.883, 0.909, 0.896]
Так, на k-fold разбиениях 						 точность: 0.89 со стандартным отклонением: 0.03


In [62]:
# в данном случае модель всегда определяет 0, что при дисбалансе классов сильно увеличивает точность

In [63]:
#loo = LeaveOneOut()

#loo_score_train = cross_val_score(log, x_train, y_train, cv = loo)
#loo_score_test = cross_val_score(log, x_test, y_test, cv = loo)

In [74]:
# 3) кросс-валидация для задачи классификации с дисбалансом классов с учетом стратфикации
from sklearn.model_selection import StratifiedKFold

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42, stratify=y)

log = LogisticRegression(max_iter = 10000).fit(x_train, y_train)

log_predict_train = log.predict(x_train)
log_predict_test = log.predict(x_test)

log_score_train = recall_score(log_predict_train, y_train, zero_division = 0, average=None)
log_score_test = recall_score(log_predict_test, y_test, zero_division = 0, average=None)

skf = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 42)

skf_score = cross_val_score(log, x, y, cv = skf)
skf_score = list(map(lambda x: round(x, 3).item(), skf_score))

print('recall для лог. регр. с учетом стр-ции на выборке для обучения / теста:\t', log_score_train, '/', log_score_test)
print('accuracy на k-fold разбиениях с учетом стратификации: \t\t\t', kf_score)
print("Так, на k-fold разбиениях с учетом стратификации \t\t\t точность: %0.2f со стандартным отклонением: %0.2f" % (np.array(kf_score).mean(), np.array(kf_score).std()))

recall для лог. регр. с учетом стр-ции на выборке для обучения / теста:	 [0.88961039 0.        ] / [0.89032258 0.        ]
accuracy на k-fold разбиениях с учетом стратификации: 			 [0.884, 0.929, 0.922, 0.916, 0.825, 0.883, 0.851, 0.883, 0.909, 0.896]
Так, на k-fold разбиениях с учетом стратификации 			 точность: 0.89 со стандартным отклонением: 0.03


In [75]:
# таким образом, стратификация не всегда может полностью решить проблему дисбаланса классов

In [88]:
# 4) изучение настраиваемых параметров коробочной полносвязной нейронной сети
#рассмотрим большинство имеющихся параметров для настройки слоев, компиляции и обучения полносвязной нейронной сети в keras

import tensorflow as tf
import keras
from tensorflow.keras.callbacks import EarlyStopping

#https://www.tensorflow.org/api_docs/python/tf/keras/Sequential
# https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense

model = keras.Sequential([
    keras.layers.Dense(16, activation='sigmoid', input_shape=(x.shape[1],)),
    keras.layers.Dense(
                        units = 8,
                        activation ='sigmoid', #https://www.tensorflow.org/api_docs/python/tf/keras/activations
                        use_bias=True,
                        kernel_initializer='glorot_uniform',
                        bias_initializer='zeros',
                        kernel_regularizer=None,
                        bias_regularizer=None,
                        activity_regularizer=None,
                        kernel_constraint=None,
                        bias_constraint=None,
                        lora_rank=None
                        ),
    keras.layers.Dense(1, activation='sigmoid')   
])

model.summary()

#https://www.tensorflow.org/api_docs/python/tf/keras/metrics
metrics = [
      keras.metrics.Recall(name='recall'),
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.Precision(name='precision'),
]

model.compile(
                optimizer = keras.optimizers.Adam(learning_rate = 1e-9), #https://www.tensorflow.org/api_docs/python/tf/keras/optimizers
                loss = 'binary_crossentropy', #https://www.tensorflow.org/api_docs/python/tf/keras/losses
                loss_weights=None,
                metrics = metrics,
                weighted_metrics=None,
                run_eagerly=False,
                steps_per_execution=1,
                jit_compile='auto',
                auto_scale_loss=True
                )

#https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    min_delta=0,
                    patience = 50,
                    verbose = 0,
                    mode='auto',
                    baseline=None,
                    restore_best_weights = True, #смотрим относительно лучшего результата, а не последнего
                    start_from_epoch=0
                    )

model.fit(
        x = x_train,
        y = y_train,
        batch_size = 16,
        epochs = 500,
        verbose = 0,
        callbacks = [early_stopping], 
        validation_split = 0.1,
        validation_data=None,
        shuffle=True,
        class_weight=None,
        sample_weight=None,
        initial_epoch=0,
        steps_per_epoch=None,
        validation_steps=None,
        validation_batch_size=None,
        validation_freq=1
        )

test_loss, test_recall, test_accuracy, test_precision = model.evaluate(x_test, y_test)
train_loss, train_recall, train_accuracy, train_precision = model.evaluate(x_train, y_train)
print('\nточность обучения \ теста:', train_accuracy, ' \ ', test_accuracy, '\nprecision обучения \ теста:', train_precision, ' \ ', test_precision)
print('recall обучения \ теста:', train_recall, ' \ ', test_recall)

D:\zmei\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                     │ (None, 16)                  │             128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 273 (1.07 KB)

 Trainable params: 273 (1.07 KB)

 Non-trainable params: 0 (0.00 B)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1226 - loss: 0.7130 - precision: 0.1111 - recall: 1.0000 
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1140 - loss: 0.7139 - precision: 0.1062 - recall: 0.9477 

точность обучения \ теста: 0.11399711668491364  \  0.1225806474685669 
precision обучения \ теста: 0.10622710734605789  \  0.1111111119389534
recall обучения \ теста: 0.9477124214172363  \  1.0


In [91]:
model.predict(x_train).flatten()

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 652us/step


array([0.5133024 , 0.51330245, 0.51329434, ..., 0.51329607, 0.51330245,
       0.51330245], dtype=float32)

In [92]:
# таким образом, даже не проводя кросс-валидацию видно, что проблема сохраняется
# изменяя параметр learning_rate можно сделать модель, предсказывающую либо только 1, либо только 0
# при learning_rate = 1e-9 получаем только 1, при больших значениях - только 0